# Notebook 05 — ML Preparation

**Project:** Financial Fraud Detection  
**Author:** Sarva  
**Date:** September 2026  

---

## Objective

Prepare the feature-engineered dataset for machine-learning by performing a **stratified train/test split** and building a **scikit-learn preprocessing pipeline**.

### Scope
| Area | Details |
|---|---|
| **Input** | `data/processed/feature_engineered_fraud_dataset.csv` |
| **Outputs** | `data/processed/X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv` |
| **Pipeline** | `models/preprocessor.joblib` |
| **Key Steps** | Identifier exclusion · Feature / target separation · Stratified split · Column transformer · Verification |
| **Excluded Steps** | Model training, SMOTE, hyperparameter tuning, performance metrics |

> **Leakage policy:** The preprocessor is **fit only on training data** and then applied to the test set to prevent data leakage.

---
## 1 · Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib

warnings.filterwarnings('ignore')

print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")
import sklearn; print(f"sklearn : {sklearn.__version__}")

pandas  : 3.0.5
numpy   : 2.4.6
sklearn : 1.5.2


---
## 2 · Load Feature-Engineered Dataset

In [2]:
# Dynamic, project-relative path — works regardless of where the notebook is launched from
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))

INPUT_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'feature_engineered_fraud_dataset.csv')
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
MODELS_DIR = os.path.join(PROJECT_ROOT, 'models')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Input file   : {INPUT_PATH}")
print(f"File exists  : {os.path.exists(INPUT_PATH)}")

Project root : c:\Users\sarva\Desktop\financial-fraud-detection
Input file   : c:\Users\sarva\Desktop\financial-fraud-detection\data\processed\feature_engineered_fraud_dataset.csv
File exists  : True


In [3]:
df = pd.read_csv(INPUT_PATH)
print(f"Dataset loaded — {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

Dataset loaded — 50,000 rows × 24 columns


,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Date,Account_Balance,Device_Type,Location,Merchant_Category,Previous_Fraudulent_Activity,...,Year,Month,Day,Day_of_Week,Amount_Bin,Balance_to_Amount_Ratio,User_Transaction_Count,User_Avg_Transaction_Amount,User_Total_Transaction_Amount,User_Amount_Deviation
0,TXN_33553,USER_1834,39.79,POS,2023-08-14,93213.17,Laptop,Sydney,Travel,0,...,2023,8,14,0,low,2342.628047,7,106.767143,747.37,-66.977143
1,TXN_9427,USER_7875,1.19,Bank Transfer,2023-06-07,75725.25,Mobile,New York,Clothing,0,...,2023,6,7,2,micro,63634.663866,5,11.688000,58.44,-10.498000
2,TXN_199,USER_2734,28.96,Online,2023-06-20,1588.96,Tablet,Mumbai,Restaurants,0,...,2023,6,20,1,low,54.867403,4,55.782500,223.13,-26.822500
3,TXN_12447,USER_2617,254.32,ATM Withdrawal,2023-12-07,76807.20,Tablet,New York,Clothing,0,...,2023,12,7,3,very_high,302.010066,9,161.852222,1456.67,92.467778
4,TXN_39489,USER_2014,31.28,POS,2023-11-11,92354.66,Mobile,Mumbai,Electronics,1,...,2023,11,11,5,low,2952.514706,9,71.240000,641.16,-39.960000


---
## 3 · Data Verification

In [4]:
# 3.1 Shape & Columns
print(f"Shape: {df.shape}")
print(f"\nColumns ({df.shape[1]}):\n{df.columns.tolist()}")

# 3.2 Data Types
print(f"\nData Types:\n{df.dtypes}")

Shape: (50000, 24)

Columns (24):
['Transaction_ID', 'User_ID', 'Transaction_Amount', 'Transaction_Type', 'Date', 'Account_Balance', 'Device_Type', 'Location', 'Merchant_Category', 'Previous_Fraudulent_Activity', 'Daily_Transaction_Count', 'Card_Type', 'Card_Age', 'Fraud_Label', 'Year', 'Month', 'Day', 'Day_of_Week', 'Amount_Bin', 'Balance_to_Amount_Ratio', 'User_Transaction_Count', 'User_Avg_Transaction_Amount', 'User_Total_Transaction_Amount', 'User_Amount_Deviation']

Data Types:
Transaction_ID                       str
User_ID                              str
Transaction_Amount               float64
Transaction_Type                     str
Date                                 str
Account_Balance                  float64
Device_Type                          str
Location                             str
Merchant_Category                    str
Previous_Fraudulent_Activity       int64
Daily_Transaction_Count            int64
Card_Type                            str
Card_Age            

In [5]:
# 3.3 Missing Values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({'Missing': missing, 'Pct (%)': missing_pct})
print(f"Total missing values: {missing.sum()}")
missing_summary[missing_summary['Missing'] > 0]

Total missing values: 0


,Missing,Pct (%)


In [6]:
# 3.4 Target Distribution
target_dist = df['Fraud_Label'].value_counts()
target_pct  = df['Fraud_Label'].value_counts(normalize=True).mul(100).round(2)

print("Fraud_Label Distribution:")
print(f"  0 (Legitimate) : {target_dist[0]:>6,}  ({target_pct[0]:.2f}%)")
print(f"  1 (Fraud)      : {target_dist[1]:>6,}  ({target_pct[1]:.2f}%)")
print(f"  Imbalance ratio: 1 : {target_dist[0] / target_dist[1]:.1f}")

Fraud_Label Distribution:
  0 (Legitimate) : 33,933  (67.87%)
  1 (Fraud)      : 16,067  (32.13%)
  Imbalance ratio: 1 : 2.1


---
## 4 · Define Target & Features

### 4.1 Target Variable

In [7]:
TARGET = 'Fraud_Label'
print(f"Target column: {TARGET}")
print(f"Unique values: {df[TARGET].unique()}")

Target column: Fraud_Label
Unique values: [0 1]


### 4.2 Exclude Identifiers

`Transaction_ID` and `User_ID` are **row-level identifiers** — they carry no predictive information about whether a transaction is fraudulent. Including them would:

1. **Cause overfitting** — the model could memorise specific IDs seen during training, producing artificially high training accuracy that does not generalise.
2. **Leak identity information** — user IDs could act as a proxy for individual fraud history, introducing a subtle form of data leakage.
3. **Inflate feature dimensionality** — one-hot-encoding thousands of unique IDs would create a sparse, uninformative feature space.

Therefore, both columns are **excluded** from the model feature set.

In [8]:
EXCLUDE_COLS = ['Transaction_ID', 'User_ID']

# Also exclude the Date column (already decomposed into Year, Month, Day, Day_of_Week)
if 'Date' in df.columns:
    EXCLUDE_COLS.append('Date')

print(f"Excluded columns: {EXCLUDE_COLS}")
print(f"  Transaction_ID unique values : {df['Transaction_ID'].nunique():,}")
print(f"  User_ID unique values        : {df['User_ID'].nunique():,}")

Excluded columns: ['Transaction_ID', 'User_ID', 'Date']
  Transaction_ID unique values : 50,000
  User_ID unique values        : 8,963


### 4.3 Define Numerical & Categorical Features

In [9]:
NUMERICAL_FEATURES = [
    'Transaction_Amount',
    'Account_Balance',
    'Previous_Fraudulent_Activity',
    'Daily_Transaction_Count',
    'Card_Age',
    'Year',
    'Month',
    'Day',
    'Day_of_Week',
    'Balance_to_Amount_Ratio',
    'User_Transaction_Count',
    'User_Avg_Transaction_Amount',
    'User_Total_Transaction_Amount',
    'User_Amount_Deviation',
]

CATEGORICAL_FEATURES = [
    'Transaction_Type',
    'Device_Type',
    'Location',
    'Merchant_Category',
    'Card_Type',
    'Amount_Bin',
]

print(f"Numerical features ({len(NUMERICAL_FEATURES)}):")
for i, col in enumerate(NUMERICAL_FEATURES, 1):
    print(f"  {i:>2}. {col}")

print(f"\nCategorical features ({len(CATEGORICAL_FEATURES)}):")
for i, col in enumerate(CATEGORICAL_FEATURES, 1):
    print(f"  {i}. {col}  —  {df[col].nunique()} unique values")

Numerical features (14):
   1. Transaction_Amount
   2. Account_Balance
   3. Previous_Fraudulent_Activity
   4. Daily_Transaction_Count
   5. Card_Age
   6. Year
   7. Month
   8. Day
   9. Day_of_Week
  10. Balance_to_Amount_Ratio
  11. User_Transaction_Count
  12. User_Avg_Transaction_Amount
  13. User_Total_Transaction_Amount
  14. User_Amount_Deviation

Categorical features (6):
  1. Transaction_Type  —  4 unique values
  2. Device_Type  —  3 unique values
  3. Location  —  5 unique values
  4. Merchant_Category  —  5 unique values
  5. Card_Type  —  4 unique values
  6. Amount_Bin  —  6 unique values


### 4.4 Validate Feature Lists

In [10]:
ALL_FEATURES = NUMERICAL_FEATURES + CATEGORICAL_FEATURES

# Verify no overlap between numerical and categorical
overlap = set(NUMERICAL_FEATURES) & set(CATEGORICAL_FEATURES)
assert len(overlap) == 0, f"Overlap between num/cat features: {overlap}"

# Verify target is NOT in feature lists
assert TARGET not in ALL_FEATURES, "Target found in features — leakage risk!"

# Verify identifiers are NOT in feature lists
for col in EXCLUDE_COLS:
    assert col not in ALL_FEATURES, f"Excluded column '{col}' found in features!"

# Verify all feature columns exist in the dataframe
missing_cols = [c for c in ALL_FEATURES if c not in df.columns]
assert len(missing_cols) == 0, f"Missing columns in dataset: {missing_cols}"

print(f"✅ Validation passed")
print(f"   Total model features : {len(ALL_FEATURES)}")
print(f"   Numerical            : {len(NUMERICAL_FEATURES)}")
print(f"   Categorical          : {len(CATEGORICAL_FEATURES)}")
print(f"   Target               : {TARGET}")
print(f"   Excluded             : {EXCLUDE_COLS}")

✅ Validation passed
   Total model features : 20
   Numerical            : 14
   Categorical          : 6
   Target               : Fraud_Label
   Excluded             : ['Transaction_ID', 'User_ID', 'Date']


---
## 5 · Feature / Target Separation

In [11]:
X = df[ALL_FEATURES].copy()
y = df[TARGET].copy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nX columns: {X.columns.tolist()}")
print(f"\n'{TARGET}' is NOT in X: {TARGET not in X.columns}")

X shape: (50000, 20)
y shape: (50000,)

X columns: ['Transaction_Amount', 'Account_Balance', 'Previous_Fraudulent_Activity', 'Daily_Transaction_Count', 'Card_Age', 'Year', 'Month', 'Day', 'Day_of_Week', 'Balance_to_Amount_Ratio', 'User_Transaction_Count', 'User_Avg_Transaction_Amount', 'User_Total_Transaction_Amount', 'User_Amount_Deviation', 'Transaction_Type', 'Device_Type', 'Location', 'Merchant_Category', 'Card_Type', 'Amount_Bin']

'Fraud_Label' is NOT in X: True


---
## 6 · Stratified Train / Test Split

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Train set : {X_train.shape[0]:>6,} rows  ({X_train.shape[0]/len(df)*100:.0f}%)")
print(f"Test set  : {X_test.shape[0]:>6,} rows  ({X_test.shape[0]/len(df)*100:.0f}%)")
print(f"Total     : {X_train.shape[0] + X_test.shape[0]:>6,} rows")

Train set : 40,000 rows  (80%)
Test set  : 10,000 rows  (20%)
Total     : 50,000 rows


In [13]:
# Verify target distribution preserved by stratification
train_dist = y_train.value_counts(normalize=True).mul(100).round(2)
test_dist  = y_test.value_counts(normalize=True).mul(100).round(2)

split_comparison = pd.DataFrame({
    'Overall (%)': df[TARGET].value_counts(normalize=True).mul(100).round(2),
    'Train (%)': train_dist,
    'Test (%)': test_dist,
})
split_comparison.index.name = TARGET

print("Target distribution preserved by stratification:\n")
print(split_comparison.to_string())
print("\n✅ Stratification successfully preserved the class distribution.")

Target distribution preserved by stratification:

             Overall (%)  Train (%)  Test (%)
Fraud_Label                                  
0                  67.87      67.86     67.87
1                  32.13      32.14     32.13

✅ Stratification successfully preserved the class distribution.


---
## 7 · Build Preprocessing Pipeline

**Numerical pipeline:**
1. `SimpleImputer(strategy='median')` — handles any remaining missing values with the median, robust to outliers.  
2. `StandardScaler()` — centres features to zero mean and unit variance.

**Categorical pipeline:**
1. `SimpleImputer(strategy='most_frequent')` — fills missing categories with the mode.  
2. `OneHotEncoder(handle_unknown='ignore')` — creates binary columns for each category; unseen categories at inference time are encoded as all zeros.

In [14]:
numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_pipeline, NUMERICAL_FEATURES),
        ('cat', categorical_pipeline, CATEGORICAL_FEATURES),
    ],
    remainder='drop',
    verbose_feature_names_out=True
)

print("Numerical pipeline:\n", numerical_pipeline)
print("\nCategorical pipeline:\n", categorical_pipeline)
print("\nColumnTransformer:\n", preprocessor)

Numerical pipeline:
 Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

Categorical pipeline:
 Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder',
                 OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

ColumnTransformer:
 ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['Transaction_Amount', 'Account_Balance',
                                  'Previous_Fraudulent_Activity',
                                  'Daily_Transaction_Count', 'Card_Age', 'Year',
                                  'Month', 'Day', 'Day_of_Week',
                                  'Balance_to_Amount_Ratio',
                               

---
## 8 · Fit & Transform

The preprocessor is **fit exclusively on `X_train`** so that scaling parameters (mean, std) and encoding vocabularies are derived only from training observations. This prevents information from the test set from influencing the transformation — a critical step to avoid **data leakage**.

In [15]:
# Fit on training data ONLY
preprocessor.fit(X_train)
print("✅ Preprocessor fitted on X_train only.")

✅ Preprocessor fitted on X_train only.


In [16]:
# Get feature names and transform both sets
feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(
    preprocessor.transform(X_train),
    columns=feature_names,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    preprocessor.transform(X_test),
    columns=feature_names,
    index=X_test.index
)

print(f"X_train transformed : {X_train_processed.shape}")
print(f"X_test  transformed : {X_test_processed.shape}")
print(f"Feature count       : {len(feature_names)}")

print(f"\nTransformed feature names ({len(feature_names)}):")
for i, name in enumerate(feature_names, 1):
    print(f"  {i:>3}. {name}")

X_train transformed : (40000, 41)
X_test  transformed : (10000, 41)
Feature count       : 41

Transformed feature names (41):
    1. num__Transaction_Amount
    2. num__Account_Balance
    3. num__Previous_Fraudulent_Activity
    4. num__Daily_Transaction_Count
    5. num__Card_Age
    6. num__Year
    7. num__Month
    8. num__Day
    9. num__Day_of_Week
   10. num__Balance_to_Amount_Ratio
   11. num__User_Transaction_Count
   12. num__User_Avg_Transaction_Amount
   13. num__User_Total_Transaction_Amount
   14. num__User_Amount_Deviation
   15. cat__Transaction_Type_ATM Withdrawal
   16. cat__Transaction_Type_Bank Transfer
   17. cat__Transaction_Type_Online
   18. cat__Transaction_Type_POS
   19. cat__Device_Type_Laptop
   20. cat__Device_Type_Mobile
   21. cat__Device_Type_Tablet
   22. cat__Location_London
   23. cat__Location_Mumbai
   24. cat__Location_New York
   25. cat__Location_Sydney
   26. cat__Location_Tokyo
   27. cat__Merchant_Category_Clothing
   28. cat__Merchant_Categ

In [17]:
X_train_processed.head()

,num__Transaction_Amount,num__Account_Balance,num__Previous_Fraudulent_Activity,num__Daily_Transaction_Count,num__Card_Age,num__Year,num__Month,num__Day,num__Day_of_Week,num__Balance_to_Amount_Ratio,...,cat__Card_Type_Amex,cat__Card_Type_Discover,cat__Card_Type_Mastercard,cat__Card_Type_Visa,cat__Amount_Bin_extreme,cat__Amount_Bin_high,cat__Amount_Bin_low,cat__Amount_Bin_medium,cat__Amount_Bin_micro,cat__Amount_Bin_very_high
3133,-0.977403,1.603525,-0.331385,-0.369026,-0.738640,0.0,0.138186,0.372471,-0.507052,0.331766,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
5433,-0.231783,0.699585,3.017634,-1.360450,-1.144301,0.0,-1.604571,-0.082573,1.493845,-0.042176,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
7621,-0.910014,-1.545712,-0.331385,-0.121171,-0.579274,0.0,-0.733192,0.144949,-1.507501,-0.045151,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
22601,0.752375,1.683443,-0.331385,0.622397,1.564934,0.0,1.009565,1.168798,-0.006828,-0.045765,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
40066,-0.966795,-1.541124,-0.331385,0.126685,-0.260540,0.0,-1.604571,-0.196334,0.993621,-0.034830,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


---
## 9 · Post-Transformation Verification

In [18]:
# 9.1 Row Counts
assert X_train_processed.shape[0] == y_train.shape[0], "Train row mismatch!"
assert X_test_processed.shape[0]  == y_test.shape[0],  "Test row mismatch!"
assert X_train_processed.shape[0] + X_test_processed.shape[0] == len(df), "Total row mismatch!"

print(f"✅ Row counts verified")
print(f"   X_train: {X_train_processed.shape[0]:,}  |  y_train: {y_train.shape[0]:,}")
print(f"   X_test : {X_test_processed.shape[0]:,}  |  y_test : {y_test.shape[0]:,}")
print(f"   Total  : {X_train_processed.shape[0] + X_test_processed.shape[0]:,}")

# 9.2 Feature Dimensions
assert X_train_processed.shape[1] == X_test_processed.shape[1], "Feature count mismatch!"

print(f"\n✅ Feature dimensions verified")
print(f"   X_train features : {X_train_processed.shape[1]}")
print(f"   X_test  features : {X_test_processed.shape[1]}")
print(f"   Original features: {len(ALL_FEATURES)} → Transformed: {X_train_processed.shape[1]}")

# 9.3 No Missing Values
train_missing = X_train_processed.isnull().sum().sum()
test_missing  = X_test_processed.isnull().sum().sum()

assert train_missing == 0, f"X_train has {train_missing} missing values!"
assert test_missing  == 0, f"X_test has {test_missing} missing values!"

print(f"\n✅ No missing values")
print(f"   X_train missing: {train_missing}")
print(f"   X_test  missing: {test_missing}")

✅ Row counts verified
   X_train: 40,000  |  y_train: 40,000
   X_test : 10,000  |  y_test : 10,000
   Total  : 50,000

✅ Feature dimensions verified
   X_train features : 41
   X_test  features : 41
   Original features: 20 → Transformed: 41

✅ No missing values
   X_train missing: 0
   X_test  missing: 0


In [19]:
# 9.4 Target Distribution in Train / Test
print("Train target distribution:")
print(y_train.value_counts().to_frame('Count').assign(
    Pct=y_train.value_counts(normalize=True).mul(100).round(2).astype(str) + '%'
))

print("\nTest target distribution:")
print(y_test.value_counts().to_frame('Count').assign(
    Pct=y_test.value_counts(normalize=True).mul(100).round(2).astype(str) + '%'
))

Train target distribution:
             Count     Pct
Fraud_Label               
0            27146  67.86%
1            12854  32.14%

Test target distribution:
             Count     Pct
Fraud_Label               
0             6787  67.87%
1             3213  32.13%


### 9.5 Data Leakage Check

In [20]:
# 1. Verify no index overlap between train and test
index_overlap = set(X_train_processed.index) & set(X_test_processed.index)
assert len(index_overlap) == 0, f"Index overlap detected: {len(index_overlap)} shared indices!"
print(f"✅ No index overlap between train and test sets (shared indices: {len(index_overlap)})")

# 2. Verify target is not in feature columns
assert TARGET not in X_train_processed.columns, "Target found in transformed training features!"
assert TARGET not in X_test_processed.columns,  "Target found in transformed test features!"
print(f"✅ Target '{TARGET}' is not present in transformed feature matrices")

# 3. Verify identifiers are not in feature columns
for col in EXCLUDE_COLS:
    assert col not in X_train_processed.columns, f"'{col}' found in transformed features!"
print(f"✅ Identifier columns are excluded from transformed features")

# 4. Verify numerical features are approximately standardised (mean ~ 0, std ~ 1) on train
num_cols = [c for c in X_train_processed.columns if c.startswith('num__')]
train_means = X_train_processed[num_cols].mean().abs()
train_stds  = X_train_processed[num_cols].std(ddof=0)  # ddof=0 to match StandardScaler
assert train_means.max() < 1e-6, "Numerical features not properly centred on train!"

# Exclude zero-variance features (e.g., Year when all values are identical)
non_const_stds = train_stds[train_stds > 0]
const_cols = train_stds[train_stds == 0].index.tolist()
if const_cols:
    print(f"   Note: Zero-variance features (constant after scaling): {const_cols}")
assert (non_const_stds - 1.0).abs().max() < 0.01, "Numerical features not properly scaled on train!"
print(f"✅ Numerical features are properly standardised on training data")

print("\nAll leakage checks passed.")

✅ No index overlap between train and test sets (shared indices: 0)
✅ Target 'Fraud_Label' is not present in transformed feature matrices
✅ Identifier columns are excluded from transformed features
   Note: Zero-variance features (constant after scaling): ['num__Year']
✅ Numerical features are properly standardised on training data

All leakage checks passed.


---
## 10 · Save Processed Datasets

In [21]:
X_train_path = os.path.join(OUTPUT_DIR, 'X_train.csv')
X_test_path  = os.path.join(OUTPUT_DIR, 'X_test.csv')
y_train_path = os.path.join(OUTPUT_DIR, 'y_train.csv')
y_test_path  = os.path.join(OUTPUT_DIR, 'y_test.csv')

X_train_processed.to_csv(X_train_path, index=False)
X_test_processed.to_csv(X_test_path, index=False)
y_train.to_csv(y_train_path, index=False)
y_test.to_csv(y_test_path, index=False)

print("✅ Datasets saved:")
for path in [X_train_path, X_test_path, y_train_path, y_test_path]:
    size_kb = os.path.getsize(path) / 1024
    print(f"   {os.path.basename(path):20s} — {size_kb:>8,.1f} KB")

✅ Datasets saved:
   X_train.csv          — 14,474.1 KB
   X_test.csv           —  3,619.3 KB
   y_train.csv          —    117.2 KB
   y_test.csv           —     29.3 KB


---
## 11 · Save Preprocessing Pipeline

In [22]:
preprocessor_path = os.path.join(MODELS_DIR, 'preprocessor.joblib')

joblib.dump(preprocessor, preprocessor_path)

size_kb = os.path.getsize(preprocessor_path) / 1024
print(f"✅ Preprocessor saved:")
print(f"   Path : {preprocessor_path}")
print(f"   Size : {size_kb:.1f} KB")

✅ Preprocessor saved:
   Path : c:\Users\sarva\Desktop\financial-fraud-detection\models\preprocessor.joblib
   Size : 6.0 KB


---
## 12 · Reload Verification

In [23]:
# Reload all saved CSV files
X_train_reloaded = pd.read_csv(X_train_path)
X_test_reloaded  = pd.read_csv(X_test_path)
y_train_reloaded = pd.read_csv(y_train_path)
y_test_reloaded  = pd.read_csv(y_test_path)

print("Reloaded shapes:")
print(f"  X_train : {X_train_reloaded.shape}")
print(f"  X_test  : {X_test_reloaded.shape}")
print(f"  y_train : {y_train_reloaded.shape}")
print(f"  y_test  : {y_test_reloaded.shape}")

# Verify shapes and values match
assert X_train_reloaded.shape == X_train_processed.shape, "X_train shape mismatch after reload!"
assert X_test_reloaded.shape  == X_test_processed.shape,  "X_test shape mismatch after reload!"
assert y_train_reloaded.shape[0] == y_train.shape[0],     "y_train row count mismatch after reload!"
assert y_test_reloaded.shape[0]  == y_test.shape[0],      "y_test row count mismatch after reload!"

assert np.allclose(X_train_reloaded.values, X_train_processed.values, atol=1e-6), \
    "X_train values changed after reload!"
assert np.allclose(X_test_reloaded.values, X_test_processed.values, atol=1e-6), \
    "X_test values changed after reload!"

print("\n✅ All CSV files reload successfully with matching shapes and values.")

Reloaded shapes:
  X_train : (40000, 41)
  X_test  : (10000, 41)
  y_train : (40000, 1)
  y_test  : (10000, 1)

✅ All CSV files reload successfully with matching shapes and values.


In [24]:
# Reload preprocessor and verify it produces the same output
preprocessor_reloaded = joblib.load(preprocessor_path)

X_test_verify = preprocessor_reloaded.transform(X_test)

assert np.allclose(X_test_verify, X_test_processed.values, atol=1e-6), \
    "Reloaded preprocessor produces different results!"

print("✅ Preprocessor reloaded and verified — produces identical transformations.")

✅ Preprocessor reloaded and verified — produces identical transformations.


---
## 13 · Summary

### What Was Done

| Step | Detail |
|---|---|
| **Input** | `feature_engineered_fraud_dataset.csv` (feature-engineered dataset) |
| **Identifiers excluded** | `Transaction_ID`, `User_ID` (+ `Date`) |
| **Features** | 14 numerical + 6 categorical = 20 raw features |
| **Split** | 80% train / 20% test, stratified by `Fraud_Label` |
| **Preprocessing** | Median imputation + StandardScaler (numerical) · Mode imputation + OneHotEncoding (categorical) |
| **Outputs saved** | `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`, `preprocessor.joblib` |

### Key Design Decisions

**1. Why Stratification?**  
Fraud detection datasets are inherently **imbalanced** — legitimate transactions far outnumber fraudulent ones. A random split could, by chance, allocate a disproportionate number of fraud cases to one set. Stratification ensures that both the training and test sets **preserve the original class distribution**, yielding reliable, unbiased evaluation.

**2. Why Were Identifiers Excluded?**  
`Transaction_ID` and `User_ID` are unique or near-unique identifiers that carry **no generalizable predictive signal**. Including them would let the model memorize training-set IDs (overfitting) and create a false sense of performance that would collapse on unseen data.

**3. Why Fit the Preprocessor Only on Training Data?**  
If scaling statistics (mean, standard deviation) or encoding vocabularies were computed on the **full dataset**, information from the test set would leak into the training pipeline. This would make model evaluation **overly optimistic** and unreliable. Fitting only on `X_train` simulates real-world deployment where future data is unseen.

**4. Why `OneHotEncoder(handle_unknown='ignore')`?**  
In production, the model may encounter **new categories** not present in the training data (e.g., a new merchant category or location). Setting `handle_unknown='ignore'` ensures such unseen categories are safely encoded as all-zero vectors rather than crashing the pipeline, making the system **robust to distribution shifts**.

---

**Next →** Notebook 06: Model Training